# Module 6 — Mini-projet : l'assistant support RAG

## Objectif
Assembler les briques : ingestion → question → **réponse ancrée + sources**, et **refus** hors base.

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070, SKEXP0001
using Microsoft.Extensions.AI;            // IEmbeddingGenerator (moderne)
using Microsoft.Extensions.VectorData;    // attributs VectorStore + collection
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.Qdrant;
using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint = new Uri("http://localhost:11434");
// Embeddings MODERNES : IEmbeddingGenerator (remplace ITextEmbeddingGenerationService)
var embed = Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text", endpoint)
    .Build().GetRequiredService<IEmbeddingGenerator<string, Embedding<float>>>();
var store = new QdrantVectorStore(new QdrantClient("localhost"), ownsClient: true);
ulong Id(string s) => BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)), 0);

In [ ]:
// Un enregistrement Qdrant : clé + texte + vecteur (768 = dimension de nomic-embed-text)
public sealed class Doc {
    [VectorStoreKey] public ulong Id { get; set; }
    [VectorStoreData] public string Text { get; set; } = "";
    [VectorStoreData(IsIndexed = true)] public string Source { get; set; } = "";
    [VectorStoreVector(768, DistanceFunction = DistanceFunction.CosineSimilarity)]
    public ReadOnlyMemory<float> Vector { get; set; }
}

In [ ]:
var kernel = Kernel.CreateBuilder().AddOllamaChatCompletion("gemma3:4b", endpoint).Build();
var collection = store.GetCollection<ulong, Doc>("assistant_kb");
await collection.EnsureCollectionExistsAsync();

// 1) Ingestion
var dir = Path.Combine("..", "data", "support_kb");
foreach (var file in Directory.GetFiles(dir, "*.md")) {
    var src = Path.GetFileNameWithoutExtension(file);
    foreach (var para in File.ReadAllText(file).Split("\n\n", StringSplitOptions.RemoveEmptyEntries)) {
        var v = (await embed.GenerateAsync(para.Trim())).Vector;
        await collection.UpsertAsync(new Doc { Id = Id(para), Text = para.Trim(), Source = src, Vector = v });
    }
}
Console.WriteLine("Base indexée.");

In [ ]:
async Task Ask(string q) {
    var qv = (await embed.GenerateAsync(q)).Vector;
    var passages = new List<string>(); var sources = new HashSet<string>();
    await foreach (var r in collection.SearchAsync(qv, top: 3)) {
        passages.Add(r.Record.Text);
        if (!string.IsNullOrWhiteSpace(r.Record.Source)) sources.Add(r.Record.Source);
    }
    var ctx = passages.Count == 0 ? "(aucun contexte)" : string.Join("\n", passages);
    var rep = await kernel.InvokePromptAsync(
        "Réponds UNIQUEMENT depuis le contexte, sinon dis \"Je ne trouve pas cette information\".\nContexte:\n{{$c}}\nQuestion: {{$q}}",
        new KernelArguments { ["c"] = ctx, ["q"] = q });
    Console.WriteLine($"Q: {q}\nR: {rep}\nSources: {string.Join(", ", sources)}\n");
}

await Ask("Mon wifi est instable");
await Ask("J'ai oublie mon mot de passe");
await Ask("Quelle est la capitale du Japon ?");   // hors base -> refus attendu

**À observer** : les 2 premières réponses sont sourcées ; la 3e (hors base) est **refusée**.

## Limites & bonnes pratiques
- Qualité du chunking = qualité du RAG. Hallucination résiduelle → consigne d'honnêteté + sources. Évaluer tôt.

## Et après ? (formation avancée)
Agents & outils, multi-agents, monitoring (Langfuse), UI Blazor.

**Bravo — assistant RAG 100 % local, en .NET, sans API dépréciée.**